# Dog Bounding Box Detection
Uses a pre-trained YOLOv5 model to detect dogs and draw bounding boxes.

In [2]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2
import numpy as np
from audio_utils import load_data, spectrogram, plot_spectrogram
from scipy.optimize import linear_sum_assignment
import os
import subprocess
import warnings
import torch
import torchreid
import torchvision.transforms as T
from scenedetect import open_video, SceneManager
from scenedetect.detectors import AdaptiveDetector
import json
import numpy as np
import pandas as pd
from pathlib import Path

/supernova/data/home/aryah/yolo_env/lib/python3.12/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
I0000 00:00:1781793346.617752 1340480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781793347.107761 1340480 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781793350.401009 1340480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off er

### Methods

In [3]:
class AppearanceExtractor:
     # Standard ReID pre-processing
    _transform = T.Compose([
        T.ToPILImage(),
        T.Resize((256, 128)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, model_name='osnet_x0_25', device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        self.device = torch.device(device)

        self.model = torchreid.models.build_model(
            model_name,
            num_classes=1,
            pretrained=True
        )

        self.model.eval()
        self.model.to(self.device)
    
    @torch.no_grad() 
    def __call__(self, frame_bgr, boxes):
        """
        frame_bgr: HxWx3 BGR image
        boxes: list of [x1, y1, x2, y2] bounding boxes

        Returns: Nx512 tensor of appearance features
        """
        if not boxes:
            return torch.empty((0, 512), dtype=torch.float32)
        
        h, w = frame_bgr.shape[:2]
        tensors = []

        for box in boxes:
            x1, y1, x2, y2 = box

            x1 = int(max(0, x1))
            y1 = int(max(0, y1))
            x2 = int(min(w, x2))
            y2 = int(min(h, y2))


            if x2 <= x1 or y2 <= y1:
                tensors.append(torch.zeros((3, 256, 128), dtype=torch.float32))  # Empty tensor for invalid box
                continue

            crop = frame_bgr[y1:y2, x1:x2]
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            tensors.append(self._transform(crop_rgb))

        batch = torch.stack(tensors).to(self.device)
        embeddings = self.model(batch)

        norms = torch.norm(embeddings, dim=1, keepdim=True).clamp(min=1e-6)
        embeddings = (embeddings / norms).cpu().numpy()

        return embeddings.astype(np.float32)

In [4]:
# Returns the most likely barking dog for a set of MO results
def mo_to_bounding_box(mo_results, stable_results):
    if len(mo_results) == 0 or len(stable_results) == 0:
        return None, None, None

    activities = []
    instabilities = []
    inds = []

    for ind, scores in mo_results.items():
        inds.append(ind)
        activities.append(scores["activity"])
        instabilities.append(scores["instability"])

    activities = np.array(activities)
    instabilities = np.array(instabilities)

    scores = 2.0 * activities + 1.0 * instabilities

    idx = np.argmax(scores)
    best_ind = inds[idx]
    best_score = scores[idx]

    print(f"- Best MO result: {best_ind}, Score: {best_score}")

    ind = best_ind if best_score > 3 else None
        
    if ind is not None:
        best = mo_results[ind]
    else:
        return None

    # Find the closest overlapping stable box
    mouth_box = np.array([
        best["left"],
        best["top"],
        best["right"],
        best["bottom"]
    ])

    best_iou = -1
    best_stable_id = None

    for stable_box, stable_id in stable_results:
        iou = box_iou(mouth_box, stable_box)

        if iou > best_iou:
            best_iou = iou
            best_stable_id = stable_id

    return best_stable_id

# Slice pose data for a specific timestamp
def cut_dataframe(df, fps, timestamp):
    frame_id = int(timestamp * fps)

    start = max(0, frame_id - 5)
    end = frame_id + 5

    slice = df.iloc[start:end]
    return slice

# Calculate IoU of two bounding boxes
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, ix2, iy1, iy2 = max(ax1, bx1), min(ax2, bx2), max(ay1, by1), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)

    intersection = (iw * ih) if iw > 0 and ih > 0 else 0
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - intersection

    return intersection / union if union > 0 else 0

# Generate bounding box color
def get_color(stable_id):
    box_colors = [
        (255, 0, 0),
        (0, 0, 255),
        (255, 255, 0),
        (255, 0, 255),
        (0, 255, 255),
        (128, 0, 255),
        (255, 128, 0),
    ]
    
    return box_colors[stable_id % len(box_colors)]

# Collect bark events
def collect_bark_events(video_path):
    s, framerate = load_data(video_path)

    freqs, times, power, log_spec = spectrogram(video_path, s, framerate)

    low_f = 300
    high_f = 3000

    band_mask = (freqs >= low_f) & (freqs <= high_f)
    band_energy = log_spec[band_mask].mean(axis=0)

    energy = (band_energy - np.mean(band_energy)) / (np.std(band_energy) + 1e-6)

    threshold = 1
    bark_frames = np.where(energy > threshold)[0]

    bark_events = []
    current = []

    for idx in bark_frames:
        if not current or idx - current[-1] <= 2:
            current.append(idx)
        else:
            bark_events.append(current)
            current = [idx]

    if current:
        bark_events.append(current)

    bark_spec_ids = [int(np.mean(event)) for event in bark_events]
    return times, log_spec, bark_spec_ids

# Helper to get spectrogram column for time t
def get_spec_col(times, time):
    return np.argmin(np.abs(times - time))

In [5]:
# Detect transitions in the video
def detect_cut(self, frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if self.prev_gray is None:
        self.prev_gray = gray
        return False

    diff = np.mean(cv2.absdiff(gray, self.prev_gray))
    self.prev_gray = gray

    return diff > 40

# Stable tracker class with Kalman filter and combined IoU + appearance matching
def create_kalman_filter():
    kf = cv2.KalmanFilter(6, 4)

    kf.transitionMatrix = np.array([
        [1,0,0,0,1,0],
        [0,1,0,0,0,1],
        [0,0,1,0,0,1],
        [0,0,0,1,0,1],
        [0,0,0,0,1,0],
        [0,0,0,0,0,1],
    ], dtype=np.float32)

    kf.measurementMatrix = np.eye(4, 6, dtype=np.float32)
    kf.processNoiseCov = np.eye(6, dtype=np.float32) * 0.5
    kf.measurementNoiseCov = np.eye(4, dtype=np.float32) * 0.05
    kf.errorCovPost = np.eye(6, dtype=np.float32)

    return kf

def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    x, y, w, h = (x1 + x2) / 2, (y1 + y2) / 2, (x2 - x1), (y2 - y1)
    return np.array([[x], [y], [w], [h]], dtype=np.float32)

class StableTracker:
    def __init__(self, iou_threshold, max_age, appearance_extractor, appearance_threshold, appearance_ema, iou_weight=0.6):
        self.next_stable_id = 0
        self.active = {}
        self.lost = {}
        self.yolo_to_stable = {}
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.extractor = appearance_extractor
        self.appearance_threshold = appearance_threshold
        self.appearance_ema = appearance_ema
        self.iou_weight = iou_weight  # α: weight for IoU vs appearance in combined cost
        self.after_scene_change = 0
        self.colors = {}

    def cosine(self, a, b):
        return float(np.dot(a, b))

    def update_embedding(self, stable_id, new_embedding, store):
        alpha = self.appearance_ema
        old = store[stable_id].get('embedding')

        if old is None:
            store[stable_id]['embedding'] = new_embedding
        else:
            blended = alpha * new_embedding + (1 - alpha) * old
            norm = np.linalg.norm(blended)
            store[stable_id]['embedding'] = blended / norm if norm > 0 else blended

    def update(self, detections, frame_id, frame_bgr=None, scene_change=False):
        if scene_change:
            self.after_scene_change = 15  # Boost appearance matching for next few frames
            self.yolo_to_stable.clear()
        if self.extractor is not None and frame_bgr is not None and detections:
            boxes = [box for box, _ in detections]
            detection_embeddings = self.extractor(frame_bgr, boxes)
        else:
            detection_embeddings = [None] * len(detections)

        updated_this_frame = set()
        unmatched_detections = []
        unmatched_embeddings = []
        matched_stable_ids = set()

        # Stage 1: Match by YOLO ID — cheap, reliable, keep as-is
        for (box, yolo_id), emb in zip(detections, detection_embeddings):
            if yolo_id in self.yolo_to_stable:
                stable_id = self.yolo_to_stable[yolo_id]
                existing = self.active.get(stable_id)

                if existing is not None and box_iou(box, existing['box']) < 0.5:
                    del self.yolo_to_stable[yolo_id]
                    unmatched_detections.append((box, yolo_id))
                    unmatched_embeddings.append(emb)
                    continue

                if stable_id in self.lost:
                    self.active[stable_id] = self.lost.pop(stable_id)
                if stable_id in self.active:
                    kf = self.active[stable_id]['kf']
                    kf.correct(xyxy_to_xywh(box))
                    self.active[stable_id]['box'] = box
                    self.active[stable_id]['last_seen'] = frame_id
                    self.yolo_to_stable[yolo_id] = stable_id
                    if emb is not None:
                        self.update_embedding(stable_id, emb, self.active)
                    matched_stable_ids.add(stable_id)
                    updated_this_frame.add(stable_id)
            else:
                unmatched_detections.append((box, yolo_id))
                unmatched_embeddings.append(emb)

        # Stage 2: Combined IoU + appearance Hungarian match over all unmatched trackers
        if unmatched_detections:
            # Predict lost tracker positions
            pred_lost_boxes = {}
            for stable_id, data in self.lost.items():
                pred = data['kf'].predict()
                x, y, w, h = pred[:4].flatten()
                pred_lost_boxes[stable_id] = (x - w/2, y - h/2, x + w/2, y + h/2)

            # Candidates = all active (not yet matched) + all lost
            candidate_ids = (
                [sid for sid in self.active if sid not in matched_stable_ids] +
                list(self.lost.keys())
            )
            candidate_boxes = [
                pred_lost_boxes[sid] if sid in self.lost else self.active[sid]['box']
                for sid in candidate_ids
            ]
            candidate_store = {**self.active, **self.lost}

            if candidate_ids:
                α = self.iou_weight
                cost_matrix = np.ones((len(unmatched_detections), len(candidate_ids)), dtype=np.float32)

                for i, (box, _) in enumerate(unmatched_detections):
                    emb = unmatched_embeddings[i]
                    for j, (sid, cbox) in enumerate(zip(candidate_ids, candidate_boxes)):
                        iou_cost = 1 - box_iou(box, cbox)

                        cand_emb = candidate_store[sid].get('embedding')
                        if emb is not None and cand_emb is not None:
                            app_cost = 1 - self.cosine(emb, cand_emb)

                            if self.after_scene_change > 0:
                                cost_matrix[i, j] = app_cost
                            else:
                                cost_matrix[i, j] = α * iou_cost + (1 - α) * app_cost
                        else:
                            # No embeddings available — IoU only
                            cost_matrix[i, j] = iou_cost

                opt_det, opt_cand = linear_sum_assignment(cost_matrix)
                assignment = {det: cand for det, cand in zip(opt_det, opt_cand)}

                still_unmatched = []
                still_unmatched_embeddings = []

                for i, (box, yolo_id) in enumerate(unmatched_detections):
                    emb = unmatched_embeddings[i]

                    if i not in assignment:
                        still_unmatched.append((box, yolo_id))
                        still_unmatched_embeddings.append(emb)
                        continue

                    j = assignment[i]
                    sid = candidate_ids[j]
                    iou_score = box_iou(box, candidate_boxes[j])
                    cand_emb = candidate_store[sid].get('embedding')
                    app_score = self.cosine(emb, cand_emb) if emb is not None and cand_emb is not None else None

                    # Accept if IoU clears the threshold, or appearance alone is confident enough
                    if self.after_scene_change > 0:
                        accept = (
                            app_score is not None
                            and app_score >= self.appearance_threshold
                        )
                    else:
                        accept = (
                            iou_score >= self.iou_threshold
                            or (
                                app_score is not None
                                and app_score >= self.appearance_threshold
                            )
                        )

                    if accept:
                        if sid in self.lost:
                            self.active[sid] = self.lost.pop(sid)

                        prev_yolo_id = self.active[sid].get('yolo_id')
                        if prev_yolo_id is not None and prev_yolo_id in self.yolo_to_stable:
                            del self.yolo_to_stable[prev_yolo_id]

                        kf = self.active[sid]['kf']
                        kf.correct(xyxy_to_xywh(box))
                        self.active[sid]['box'] = box
                        self.active[sid]['last_seen'] = frame_id
                        self.active[sid]['yolo_id'] = yolo_id
                        self.yolo_to_stable[yolo_id] = sid
                        if emb is not None:
                            self.update_embedding(sid, emb, self.active)
                        matched_stable_ids.add(sid)
                        updated_this_frame.add(sid)
                    else:
                        still_unmatched.append((box, yolo_id))
                        still_unmatched_embeddings.append(emb)

                unmatched_detections = still_unmatched
                unmatched_embeddings = still_unmatched_embeddings

        # Stage 3: Anything still unmatched becomes a new stable ID
        for (box, yolo_id), emb in zip(unmatched_detections, unmatched_embeddings):
            stable_id = self.next_stable_id
            self.next_stable_id += 1

            kf = create_kalman_filter()
            x, y, w, h = xyxy_to_xywh(box).flatten()
            kf.statePre = np.array([[x],[y],[w],[h],[0],[0]], np.float32)
            kf.statePost = kf.statePre.copy()

            self.yolo_to_stable[yolo_id] = stable_id
            self.active[stable_id] = {
                'kf': kf,
                'box': box,
                'last_seen': frame_id,
                'yolo_id': yolo_id,
                'embedding': emb
            }
            updated_this_frame.add(stable_id)

        # Move aged-out trackers to lost
        for stable_id, data in list(self.active.items()):
            if frame_id - data['last_seen'] > self.max_age:
                self.lost[stable_id] = self.active.pop(stable_id)

        if self.after_scene_change > 0:
            self.after_scene_change -= 1

        return [
            (self.active[sid]['box'], sid)
            for sid in updated_this_frame
            if sid in self.active
        ]

In [6]:
def run_full_video_annotation_tracking(
    video_id,
    video_path,
    boxed_video_path,
    tracks_csv_path,
    model,
    appearance_model,
    appearance_threshold,
    appearance_ema,
    conf_threshold,
    iou_threshold,
    max_age,
    device,
):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Source video not found: {video_path}")

    boxed_video_path = Path(boxed_video_path)
    tracks_csv_path = Path(tracks_csv_path)
    boxed_video_path.parent.mkdir(parents=True, exist_ok=True)
    tracks_csv_path.parent.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    temp_silent = boxed_video_path.with_name(f".{boxed_video_path.stem}_silent_tmp.mp4")
    out = cv2.VideoWriter(
        str(temp_silent),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )
    if not out.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create silent video writer: {temp_silent}")

    extractor = AppearanceExtractor(appearance_model, device=device)
    stable_tracker = StableTracker(
        iou_threshold,
        max_age,
        extractor,
        appearance_threshold,
        appearance_ema,
    )

    track_rows = []
    frame_id = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            conf=conf_threshold,
            verbose=False,
            device=device,
        )
        r = results[0]

        detections = []
        conf_lookup = {}

        if r.boxes is not None and r.boxes.id is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            yolo_ids = r.boxes.id.cpu().numpy()

            for i in range(len(boxes)):
                if model.names[int(cls_ids[i])] == "dog" and confs[i] > conf_threshold:
                    yid = int(yolo_ids[i])
                    conf_lookup[yid] = float(confs[i])
                    detections.append((boxes[i], yid))

        stable_results = stable_tracker.update(
            detections, frame_id, frame, scene_change=False
        )

        stable_id_to_conf = {}
        for yolo_id, stable_id in stable_tracker.yolo_to_stable.items():
            if yolo_id in conf_lookup:
                stable_id_to_conf[int(stable_id)] = conf_lookup[yolo_id]

        draw_frame = frame.copy()
        time_sec = frame_id / fps

        for box, stable_id in stable_results:
            stable_id = int(stable_id)
            x1, y1, x2, y2 = box
            x1 = int(max(0, min(w, x1)))
            y1 = int(max(0, min(h, y1)))
            x2 = int(max(0, min(w, x2)))
            y2 = int(max(0, min(h, y2)))

            if x2 <= x1 or y2 <= y1:
                continue

            conf = stable_id_to_conf.get(stable_id, 0.0)
            color = get_color(stable_id)

            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)
            cv2.rectangle(draw_frame, (x1, y1 - 45), (x1 + 180, y1), color, -1)
            cv2.putText(
                draw_frame,
                f"dog #{stable_id} {conf:.2f}",
                (x1 + 5, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2,
            )

            track_rows.append(
                {
                    "video_id": video_id,
                    "frame": frame_id,
                    "time_sec": time_sec,
                    "track_id": stable_id,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "confidence": conf,
                    "class_name": "dog",
                }
            )

        out.write(draw_frame)
        frame_id += 1

        if frame_id % 500 == 0:
            print(f"Processed {frame_id} frames ({time_sec:.1f}s)")

    cap.release()
    out.release()

    tracks_df = pd.DataFrame(track_rows)
    tracks_df.to_csv(tracks_csv_path, index=False)
    print(f"Saved tracks CSV: {tracks_csv_path} ({len(tracks_df)} rows)")

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            str(temp_silent),
            "-i",
            str(video_path),
            "-map",
            "0:v:0",
            "-map",
            "1:a:0?",
            "-c:v",
            "copy",
            "-c:a",
            "aac",
            "-shortest",
            str(boxed_video_path),
        ],
        check=True,
    )

    temp_silent.unlink(missing_ok=True)
    print(f"Saved boxed video: {boxed_video_path}")

    return tracks_df


def make_boxed_bark_annotation_clips(
    video_id,
    segments_csv_path,
    boxed_video_path,
    tracks_csv_path,
    clips_dir,
    annotation_csv_path,
    pad_before=0.5,
    pad_after=0.5,
):
    segments_df = pd.read_csv(segments_csv_path)
    segments_df = (
        segments_df[segments_df["video_id"] == video_id]
        .sort_values(["start_time", "end_time"])
        .reset_index(drop=True)
    )

    tracks_df = pd.read_csv(tracks_csv_path)
    tracks_df["track_id"] = tracks_df["track_id"].astype(int)

    boxed_video_path = Path(boxed_video_path)
    clips_dir = Path(clips_dir)
    clips_dir.mkdir(parents=True, exist_ok=True)
    annotation_csv_path = Path(annotation_csv_path)
    annotation_csv_path.parent.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(boxed_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open boxed video: {boxed_video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_duration = frame_count / fps
    cap.release()

    def track_ids_in_range(t_start, t_end):
        mask = (tracks_df["time_sec"] >= t_start) & (tracks_df["time_sec"] <= t_end)
        ids = sorted(tracks_df.loc[mask, "track_id"].unique())
        return "|".join(str(i) for i in ids)

    rows = []
    for bark_id, seg in segments_df.iterrows():
        original_start = float(seg["start_time"])
        original_end = float(seg["end_time"])
        original_duration = original_end - original_start

        clip_start = max(0.0, original_start - pad_before)
        clip_end = min(video_duration, original_end + pad_after)
        clip_duration = clip_end - clip_start

        clip_path = clips_dir / f"{video_id}_bark_{bark_id:03d}_boxed.mp4"

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-ss",
                str(clip_start),
                "-i",
                str(boxed_video_path),
                "-t",
                str(clip_duration),
                "-c:v",
                "libx264",
                "-c:a",
                "aac",
                "-movflags",
                "+faststart",
                str(clip_path),
            ],
            check=True,
        )

        rows.append(
            {
                "video_id": video_id,
                "bark_id": bark_id,
                "source_video_path": seg["video_path"],
                "original_start_time": original_start,
                "original_end_time": original_end,
                "original_duration": original_duration,
                "clip_start_time": clip_start,
                "clip_end_time": clip_end,
                "clip_path": str(clip_path),
                "track_ids_during_bark": track_ids_in_range(
                    original_start, original_end
                ),
                "track_ids_in_clip": track_ids_in_range(clip_start, clip_end),
                "barking_track_id": "",
                "label": "",
                "notes": "",
            }
        )

    annotation_df = pd.DataFrame(rows)
    annotation_df.to_csv(annotation_csv_path, index=False)
    print(f"Saved annotation sheet: {annotation_csv_path} ({len(annotation_df)} rows)")

    return annotation_df

In [7]:
# Processing function with built-in tracker (no Kalman or appearance)
def process_video_old(video_path, output_path, log_spec, bark_spec_ids, times, model, conf_threshold):
    # Load dog video
    cap = cv2.VideoCapture(video_path)

    print("Video opened:", cap.isOpened())

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Precompute spectrogram visualization
    spec_full = np.flipud(log_spec)

    spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
    spec_vis_full = spec_vis_full.astype(np.uint8)
    spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
    spec_vis_full = cv2.resize(spec_vis_full, (w, h))

    # Video writer for output
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h)
    )

    print("Writer opened:", out.isOpened())

    frame_id = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps # Current time in seconds
        spec_id = get_spec_col(times, t)
        spec_vis = spec_vis_full.copy()
        x_pos = int(spec_id / log_spec.shape[1] * w)
        cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

        results = model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            imgsz=640,
            conf=conf_threshold,
            verbose=False
        )

        r = results[0]

        if r.boxes is not None and r.boxes.id is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            yolo_ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

            for i in range(len(boxes)):
                if model.names[int(cls_ids[i])] == 'dog' and confs[i] > conf_threshold:
                    if yolo_ids is None:
                        continue

            # for box, stable_id in stable_results:
                x1, y1, x2, y2 = map(int, boxes[i])

                color = get_color(int(yolo_ids[i]))

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 135, y1), color, -1)
                cv2.putText(frame, f'dog #{yolo_ids[i]}', 
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

        for bark_id in bark_spec_ids:
            if abs(bark_id - spec_id) < 10:  # only nearby ones
                x = int(bark_id / log_spec.shape[1] * w)

                cv2.rectangle(
                    spec_vis,
                    (x - 5, 0),
                    (x + 5, h),
                    (255, 0, 255),
                    -1
                )

        frame = frame.astype('uint8')
        frame = np.ascontiguousarray(frame)

        frame = cv2.resize(frame, (w, h))
        spec_vis = cv2.resize(spec_vis, (w, h))
        frame = np.hstack([frame, spec_vis])

        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()
    # cv2.destroyAllWindows()

    print("Processing complete")

In [8]:
import json
import uuid

# --- Track segmentation thresholds (adjustable) ---
# A stable_id's keyframe sequence is split into separate Label Studio
# videorectangle segments whenever a transition between consecutive present
# frames looks suspicious. This stops Label Studio from interpolating one box
# across an ID swap / re-acquisition / tracker jump (the "phantom"/"smeared"
# boxes that never appear in the MP4).
MAX_CENTER_JUMP_RATIO = 0.10   # max center move (fraction of video diagonal) per frame
MIN_IOU_TO_CONTINUE   = 0.15   # min IoU between consecutive boxes to stay in one segment
MAX_SIZE_CHANGE_RATIO = 2.0    # max area growth/shrink factor between consecutive boxes


def _box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def _box_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def _segment_track(present, frame_map, diagonal):
    """Split a single track's sorted frame list into contiguous, well-behaved
    segments. Returns a list of segments, each a list of frame ids.

    Starts a new segment between prev_fr and curr_fr when ANY of:
      1. frame gap > 1
      2. IoU(prev_box, curr_box) < MIN_IOU_TO_CONTINUE
      3. center jump / diagonal > MAX_CENTER_JUMP_RATIO
      4. area ratio (grow or shrink) > MAX_SIZE_CHANGE_RATIO
    """
    if not present:
        return []

    segments = [[present[0]]]
    for prev_fr, curr_fr in zip(present, present[1:]):
        prev_box = frame_map[prev_fr]
        curr_box = frame_map[curr_fr]

        frame_gap = curr_fr - prev_fr
        iou = box_iou(prev_box, curr_box)

        pcx, pcy = _box_center(prev_box)
        ccx, ccy = _box_center(curr_box)
        center_jump = ((pcx - ccx) ** 2 + (pcy - ccy) ** 2) ** 0.5
        center_ratio = center_jump / diagonal if diagonal > 0 else 0.0

        prev_area = _box_area(prev_box)
        curr_area = _box_area(curr_box)
        if prev_area > 0 and curr_area > 0:
            size_ratio = max(curr_area / prev_area, prev_area / curr_area)
        else:
            size_ratio = float("inf")  # a zero-area box is always suspicious

        if (frame_gap > 1
                or iou < MIN_IOU_TO_CONTINUE
                or center_ratio > MAX_CENTER_JUMP_RATIO
                or size_ratio > MAX_SIZE_CHANGE_RATIO):
            segments.append([curr_fr])
        else:
            segments[-1].append(curr_fr)

    return segments


def export_label_studio(stable_tracker_results_per_frame, fps, w, h, total_frames, video_url, output_path):
    """
    stable_tracker_results_per_frame: dict of {frame_id: [(box, stable_id), ...]}

    Emits one Label Studio `videorectangle` PER SEGMENT (not per stable_id). Each
    stable_id track is split into contiguous, well-behaved segments (see
    _segment_track), so Label Studio never interpolates a box across a gap, an ID
    swap, or a sudden jump/resize. Each segment's visible span is explicitly closed
    with an `enabled: False` keyframe at last_frame + 1 (reusing the last box), so no
    box persists after the dog disappears. Only boxes actually drawn into the MP4
    (i.e. present in this store) are exported.
    """

    def keyframe(frame_id, box, enabled):
        x1, y1, x2, y2 = box
        return {
            "frame": int(frame_id),
            "time": round(frame_id / fps, 3),
            "x": float(x1 / w * 100),
            "y": float(y1 / h * 100),
            "width": float((x2 - x1) / w * 100),
            "height": float((y2 - y1) / h * 100),
            "enabled": enabled,
            "rotation": 0,
        }

    diagonal = (w ** 2 + h ** 2) ** 0.5

    # Group by stable ID -> {frame_id: box}
    tracks = {}
    for frame_id, results in stable_tracker_results_per_frame.items():
        for box, stable_id in results:
            tracks.setdefault(stable_id, {})[frame_id] = box

    results = []
    debug_rows = []
    for stable_id, frame_map in tracks.items():
        present = sorted(frame_map)
        segments = _segment_track(present, frame_map, diagonal)
        debug_rows.append((stable_id, len(present), segments))

        for segment in segments:
            sequence = []
            for fr in segment:
                sequence.append(keyframe(fr, frame_map[fr], True))

            # Close the segment: a disabled keyframe right after its last frame,
            # reusing the last box. Within a segment all frames are contiguous
            # (a gap forces a split), so one closing keyframe is enough.
            last_fr = segment[-1]
            if last_fr + 1 < total_frames:
                sequence.append(keyframe(last_fr + 1, frame_map[last_fr], False))

            results.append({
                "id": str(uuid.uuid4())[:10],
                "type": "videorectangle",
                "value": {
                    "labels": ["Dog"],
                    "duration": total_frames / fps,
                    "sequence": sequence,
                    "framesCount": int(total_frames)
                },
                "origin": "manual",
                "to_name": "video",
                "from_name": "box"
            })

    output = {
        "data": {"video": video_url},
        "predictions": [{
            "model_version": "v1",
            "result": results
        }]
    }

    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)

    # --- Debug summary: per stable_id, frame count, segment count, segment spans ---
    print(f"Exported {len(tracks)} tracks as {len(results)} segments to {output_path}")
    for stable_id, n_frames, segments in sorted(debug_rows):
        spans = ", ".join(f"{seg[0]}-{seg[-1]}" for seg in segments)
        print(f"  stable_id {stable_id}: {n_frames} frames, {len(segments)} segment(s) [{spans}]")


def export_cvat_xml(stable_results_per_frame, fps, w, h, total_frames, output_path):
    """
    stable_results_per_frame: dict of {frame_id: [(box, stable_id), ...]}

    Emits CVAT for video 1.1 XML with one rectangle track per segment (see
    _segment_track). Each segment is closed with outside="1" at last_frame + 1.
    Only boxes actually drawn into the MP4 are exported.
    """
    from datetime import datetime
    from xml.etree import ElementTree as ET

    def _cvat_box(track_el, frame_id, box, outside):
        x1, y1, x2, y2 = box
        ET.SubElement(
            track_el,
            "box",
            frame=str(int(frame_id)),
            xtl=f"{float(x1):.2f}",
            ytl=f"{float(y1):.2f}",
            xbr=f"{float(x2):.2f}",
            ybr=f"{float(y2):.2f}",
            outside="1" if outside else "0",
            occluded="0",
            keyframe="1",
        )

    diagonal = (w ** 2 + h ** 2) ** 0.5

    tracks = {}
    for frame_id, results in stable_results_per_frame.items():
        for box, stable_id in results:
            tracks.setdefault(stable_id, {})[frame_id] = box

    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"

    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "id").text = "0"
    ET.SubElement(task, "name").text = "dog_tracking"
    ET.SubElement(task, "size").text = str(int(total_frames))
    ET.SubElement(task, "mode").text = "interpolation"
    ET.SubElement(task, "overlap").text = "0"
    ET.SubElement(task, "bugtracker").text = ""
    ET.SubElement(task, "flipped").text = "False"
    ET.SubElement(task, "created").text = datetime.now().isoformat()
    ET.SubElement(task, "updated").text = datetime.now().isoformat()

    labels = ET.SubElement(task, "labels")
    label = ET.SubElement(labels, "label")
    ET.SubElement(label, "name").text = "dog"
    ET.SubElement(label, "attributes")

    segments_el = ET.SubElement(task, "segments")
    segment = ET.SubElement(segments_el, "segment")
    ET.SubElement(segment, "id").text = "0"
    ET.SubElement(segment, "start").text = "0"
    ET.SubElement(segment, "stop").text = str(max(0, int(total_frames) - 1))
    ET.SubElement(segment, "url").text = ""

    original_size = ET.SubElement(task, "original_size")
    ET.SubElement(original_size, "width").text = str(int(w))
    ET.SubElement(original_size, "height").text = str(int(h))
    ET.SubElement(meta, "dumped").text = datetime.now().isoformat()

    track_id = 0
    n_segments = 0
    debug_rows = []
    for stable_id, frame_map in sorted(tracks.items()):
        present = sorted(frame_map)
        segments = _segment_track(present, frame_map, diagonal)
        debug_rows.append((stable_id, len(present), segments))

        for segment_frames in segments:
            track_el = ET.SubElement(
                root,
                "track",
                id=str(track_id),
                label="dog",
                source="manual",
            )
            for fr in segment_frames:
                _cvat_box(track_el, fr, frame_map[fr], outside=False)

            last_fr = segment_frames[-1]
            if last_fr + 1 < total_frames:
                _cvat_box(track_el, last_fr + 1, frame_map[last_fr], outside=True)

            track_id += 1
            n_segments += 1

    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ")
    tree.write(output_path, encoding="utf-8", xml_declaration=True)

    print(f"Exported {len(tracks)} stable IDs as {n_segments} CVAT track(s) to {output_path}")
    for stable_id, n_frames, segments in debug_rows:
        spans = ", ".join(f"{seg[0]}-{seg[-1]}" for seg in segments)
        print(f"  stable_id {stable_id}: {n_frames} frames, {len(segments)} segment(s) [{spans}]")


In [9]:
# Main processing function
def process_video(video_path, output_path, log_spec, bark_spec_ids, times, model, appearance_model, appearance_threshold, appearance_ema, conf_threshold, iou_threshold, max_age):
    # Load dog video
    cap = cv2.VideoCapture(video_path)

    print("Video opened:", cap.isOpened())

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    video = open_video(video_path)
    scene_manager = SceneManager()
    scene_manager.add_detector(AdaptiveDetector())

    scene_manager.detect_scenes(video)
    scenes = scene_manager.get_scene_list()

    scene_boundaries = set()
    for start, _ in scenes[1:]:
        scene_boundaries.add(start.get_frames())


    extractor = AppearanceExtractor(appearance_model)
    stable_tracker = StableTracker(iou_threshold, max_age, extractor, appearance_threshold, appearance_ema)

    print("Stable tracker initialized")

    # Precompute spectrogram visualization
    spec_full = np.flipud(log_spec)

    spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
    spec_vis_full = spec_vis_full.astype(np.uint8)
    spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
    spec_vis_full = cv2.resize(spec_vis_full, (w, h))

    # Video writer for output
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h)
    )

    print("Writer opened:", out.isOpened())

    frame_id = 0
    processed_barks = set()

    stable_results_per_frame = {}

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps # Current time in seconds
        spec_id = get_spec_col(times, t)
        spec_vis = spec_vis_full.copy()
        x_pos = int(spec_id / log_spec.shape[1] * w)
        cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

        results = model.track(frame, persist=True, verbose=False)

        r = results[0]

        if r.boxes is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            yolo_ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

            detections = []
            conf_lookup = {}

            for i in range(len(boxes)):
                if model.names[int(cls_ids[i])] == 'dog' and confs[i] > conf_threshold:
                    if yolo_ids is None:
                        continue

                    conf_lookup[yolo_ids[i]] = confs[i]
                    detections.append((boxes[i], yolo_ids[i]))

            stable_results = stable_tracker.update(detections, frame_id, frame, scene_change=(frame_id in scene_boundaries))


            for bark_id in bark_spec_ids:
                if bark_id not in processed_barks and abs(spec_id - bark_id) <= 1:
                    bark_t = times[bark_id]

                    print(f"Processing bark at frame {bark_id}, time {bark_t}")
                    
                    x = int(bark_id / log_spec.shape[1] * w)

                    cv2.rectangle(
                        spec_vis,
                        (x - 5, 0),
                        (x + 5, h),
                        (255, 0, 255),
                        -1
                    )

                    processed_barks.add(bark_id)

            stable_id_to_conf = {}

            for yolo_id, stable_id in stable_tracker.yolo_to_stable.items():
                if yolo_id in conf_lookup:
                    stable_id_to_conf[stable_id] = conf_lookup[yolo_id]

            for box, stable_id in stable_results:
                x1, y1, x2, y2 = map(int, box)
                DEBUG_FRAME = 1256
                if frame_id == DEBUG_FRAME:
                    width, height = x2 - x1, y2 - y1
                    area = width * height
                    print(frame_id, stable_id, x1, y1, x2, y2, width, height, area)
                conf = stable_id_to_conf.get(stable_id, 0)

                color = get_color(stable_id)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 135, y1), color, -1)
                cv2.putText(frame, f'dog #{stable_id} {conf:.2f}',
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

                stable_results_per_frame.setdefault(frame_id, []).append(
                    ([x1, y1, x2, y2], stable_id)
                )

        frame = frame.astype('uint8')
        frame = np.ascontiguousarray(frame)

        frame = cv2.resize(frame, (w, h))
        spec_vis = cv2.resize(spec_vis, (w, h))
        frame = np.hstack([frame, spec_vis])

        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()
    # cv2.destroyAllWindows()

    print("Processing complete")

    total_frames = frame_id

    # Expose the per-frame store so the verification cell can compare
    # stored/drawn boxes against the exported JSON.
    globals()['stable_results_per_frame'] = stable_results_per_frame

    # export_label_studio(
    #     stable_results_per_frame,
    #     fps, w, h,
    #     total_frames,
    #     video_url=f'/data/upload/{title}.mp4',
    #     output_path=f'files/{title}-labelstudio.json',
    # )

    export_cvat_xml(
        stable_results_per_frame,
        fps, w, h,
        total_frames,
        output_path=f"files/{title}-cvat.xml",
    )


### Models

In [10]:
from ultralytics import YOLO, RTDETR

In [11]:
video_folder = 'files/dataset'
output_folder = 'files/rt-outputs'

MODEL_WEIGHTS = "rtdetr-l.pt"
model = RTDETR(MODEL_WEIGHTS)
model_name = 'rt'
appearance_model = 'osnet_x0_25'
appearance_threshold = 0.7
appearance_ema = 0.3
conf_threshold = 0.6
iou_threshold = 0.08
max_age = 20

device = 0 if torch.cuda.is_available() else "cpu"

print("Model:", MODEL_WEIGHTS)
print("Device:", device)

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

Model: rtdetr-l.pt
Device: 0


## Run Boxed Bark Annotation Pipeline — Video 16

In [14]:
from datetime import datetime
from xml.etree import ElementTree as ET


def make_cvat_ready_bark_clips(
    video_id,
    segments_csv_path,
    source_video_path,
    tracks_csv_path,
    output_dir,
    manifest_csv_path,
    pad_before=0.5,
    pad_after=0.5,
    label_name="Dog",
):
    """
    For each bark timestamp:

    1. Cut a CLEAN, unboxed clip from the original source MP4.
    2. Select matching boxes from the full-video tracks CSV.
    3. Convert source-video frame numbers to clip-local frame numbers.
    4. Create a matching CVAT 1.1 XML annotation file.

    Each output pair:
        16_bark_000.mp4
        16_bark_000.xml
    """

    segments_csv_path = Path(segments_csv_path)
    source_video_path = Path(source_video_path)
    tracks_csv_path = Path(tracks_csv_path)
    output_dir = Path(output_dir)
    manifest_csv_path = Path(manifest_csv_path)

    if not segments_csv_path.exists():
        raise FileNotFoundError(
            f"Segments CSV not found: {segments_csv_path.resolve()}"
        )

    if not source_video_path.exists():
        raise FileNotFoundError(
            f"Source video not found: {source_video_path.resolve()}"
        )

    if not tracks_csv_path.exists():
        raise FileNotFoundError(
            f"Tracks CSV not found: {tracks_csv_path.resolve()}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_csv_path.parent.mkdir(parents=True, exist_ok=True)

    segments_df = pd.read_csv(segments_csv_path)

    segments_df = (
        segments_df[segments_df["video_id"] == video_id]
        .sort_values(["start_time", "end_time"])
        .reset_index(drop=True)
    )

    if segments_df.empty:
        raise ValueError(
            f"No bark segments found for video_id={video_id}"
        )

    tracks_df = pd.read_csv(tracks_csv_path)

    required_track_columns = {
        "frame",
        "time_sec",
        "track_id",
        "x1",
        "y1",
        "x2",
        "y2",
    }

    missing_columns = required_track_columns - set(tracks_df.columns)

    if missing_columns:
        raise ValueError(
            "Tracks CSV is missing columns: "
            + ", ".join(sorted(missing_columns))
        )

    tracks_df["frame"] = tracks_df["frame"].astype(int)
    tracks_df["track_id"] = tracks_df["track_id"].astype(int)

    source_cap = cv2.VideoCapture(str(source_video_path))

    if not source_cap.isOpened():
        raise RuntimeError(
            f"Could not open source video: {source_video_path}"
        )

    fps = float(source_cap.get(cv2.CAP_PROP_FPS))
    width = int(source_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(source_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(source_cap.get(cv2.CAP_PROP_FRAME_COUNT))

    source_cap.release()

    if fps <= 0:
        raise RuntimeError(
            f"Invalid FPS for source video: {fps}"
        )

    if width <= 0 or height <= 0 or total_frames <= 0:
        raise RuntimeError(
            "Invalid source video metadata: "
            f"{width}x{height}, frames={total_frames}"
        )

    def add_cvat_box(
        track_element,
        frame_number,
        box,
        outside,
    ):
        x1, y1, x2, y2 = box

        ET.SubElement(
            track_element,
            "box",
            frame=str(int(frame_number)),
            xtl=f"{float(x1):.2f}",
            ytl=f"{float(y1):.2f}",
            xbr=f"{float(x2):.2f}",
            ybr=f"{float(y2):.2f}",
            outside="1" if outside else "0",
            occluded="0",
            keyframe="1",
            z_order="0",
        )

    def create_clip_cvat_xml(
        clip_tracks,
        clip_start_frame,
        clip_total_frames,
        clip_name,
        xml_path,
    ):
        root = ET.Element("annotations")
        ET.SubElement(root, "version").text = "1.1"

        meta = ET.SubElement(root, "meta")
        task = ET.SubElement(meta, "task")

        ET.SubElement(task, "id").text = "0"
        ET.SubElement(task, "name").text = clip_name
        ET.SubElement(task, "size").text = str(
            int(clip_total_frames)
        )
        ET.SubElement(task, "mode").text = "interpolation"
        ET.SubElement(task, "overlap").text = "0"
        ET.SubElement(task, "bugtracker").text = ""
        ET.SubElement(task, "flipped").text = "False"
        ET.SubElement(task, "created").text = (
            datetime.now().isoformat()
        )
        ET.SubElement(task, "updated").text = (
            datetime.now().isoformat()
        )

        labels_element = ET.SubElement(task, "labels")
        label_element = ET.SubElement(
            labels_element,
            "label",
        )

        ET.SubElement(
            label_element,
            "name",
        ).text = label_name

        ET.SubElement(
            label_element,
            "attributes",
        )

        segments_element = ET.SubElement(task, "segments")
        segment_element = ET.SubElement(
            segments_element,
            "segment",
        )

        ET.SubElement(segment_element, "id").text = "0"
        ET.SubElement(segment_element, "start").text = "0"
        ET.SubElement(segment_element, "stop").text = str(
            max(0, int(clip_total_frames) - 1)
        )
        ET.SubElement(segment_element, "url").text = ""

        original_size = ET.SubElement(
            task,
            "original_size",
        )

        ET.SubElement(
            original_size,
            "width",
        ).text = str(width)

        ET.SubElement(
            original_size,
            "height",
        ).text = str(height)

        ET.SubElement(meta, "dumped").text = (
            datetime.now().isoformat()
        )

        for stable_id, track_group in clip_tracks.groupby(
            "track_id",
            sort=True,
        ):
            track_group = (
                track_group
                .sort_values("frame")
                .drop_duplicates(
                    subset=["frame"],
                    keep="last",
                )
            )

            track_element = ET.SubElement(
                root,
                "track",
                id=str(int(stable_id)),
                label=label_name,
                source="auto",
            )

            previous_local_frame = None
            previous_box = None

            for _, track_row in track_group.iterrows():
                source_frame = int(track_row["frame"])

                local_frame = (
                    source_frame - clip_start_frame
                )

                if (
                    local_frame < 0
                    or local_frame >= clip_total_frames
                ):
                    continue

                box = [
                    max(
                        0.0,
                        min(width, float(track_row["x1"])),
                    ),
                    max(
                        0.0,
                        min(height, float(track_row["y1"])),
                    ),
                    max(
                        0.0,
                        min(width, float(track_row["x2"])),
                    ),
                    max(
                        0.0,
                        min(height, float(track_row["y2"])),
                    ),
                ]

                if box[2] <= box[0] or box[3] <= box[1]:
                    continue

                # Close the visible section when the track
                # disappears for one or more frames.
                if (
                    previous_local_frame is not None
                    and local_frame > previous_local_frame + 1
                ):
                    outside_frame = previous_local_frame + 1

                    if outside_frame < clip_total_frames:
                        add_cvat_box(
                            track_element,
                            outside_frame,
                            previous_box,
                            outside=True,
                        )

                add_cvat_box(
                    track_element,
                    local_frame,
                    box,
                    outside=False,
                )

                previous_local_frame = local_frame
                previous_box = box

            # Close the track after its final visible frame.
            if previous_local_frame is not None:
                outside_frame = previous_local_frame + 1

                if outside_frame < clip_total_frames:
                    add_cvat_box(
                        track_element,
                        outside_frame,
                        previous_box,
                        outside=True,
                    )

        tree = ET.ElementTree(root)
        ET.indent(tree, space="  ")

        tree.write(
            xml_path,
            encoding="utf-8",
            xml_declaration=True,
        )

    manifest_rows = []

    for bark_id, segment in segments_df.iterrows():
        bark_start = float(segment["start_time"])
        bark_end = float(segment["end_time"])

        # Align clips to exact source-video frame boundaries.
        clip_start_frame = max(
            0,
            int(np.floor(
                (bark_start - pad_before) * fps
            )),
        )

        clip_end_frame = min(
            total_frames - 1,
            int(np.ceil(
                (bark_end + pad_after) * fps
            )) - 1,
        )

        if clip_end_frame < clip_start_frame:
            print(
                f"Skipping invalid bark interval {bark_id}"
            )
            continue

        expected_clip_frames = (
            clip_end_frame - clip_start_frame + 1
        )

        clip_start_time = clip_start_frame / fps
        clip_duration = expected_clip_frames / fps

        clip_stem = (
            f"{video_id}_bark_{bark_id:03d}"
        )

        raw_clip_path = (
            output_dir / f"{clip_stem}.mp4"
        )

        xml_path = (
            output_dir / f"{clip_stem}.xml"
        )

        temporary_silent_path = (
            output_dir / f".{clip_stem}_silent_tmp.mp4"
        )

        # Write exact unboxed source frames.
        cap = cv2.VideoCapture(str(source_video_path))

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not reopen source video: "
                f"{source_video_path}"
            )

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            clip_start_frame,
        )

        writer = cv2.VideoWriter(
            str(temporary_silent_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height),
        )

        if not writer.isOpened():
            cap.release()

            raise RuntimeError(
                "Could not create temporary clip: "
                f"{temporary_silent_path}"
            )

        frames_written = 0

        while frames_written < expected_clip_frames:
            success, frame = cap.read()

            if not success:
                break

            writer.write(frame)
            frames_written += 1

        cap.release()
        writer.release()

        if frames_written == 0:
            temporary_silent_path.unlink(
                missing_ok=True
            )

            print(
                f"Skipping bark {bark_id}: "
                "no frames were written"
            )
            continue

        actual_clip_end_frame = (
            clip_start_frame + frames_written - 1
        )

        actual_clip_duration = frames_written / fps

        # Attach the matching source audio to the clean clip.
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-loglevel",
                "error",
                "-i",
                str(temporary_silent_path),
                "-ss",
                f"{clip_start_time:.6f}",
                "-i",
                str(source_video_path),
                "-map",
                "0:v:0",
                "-map",
                "1:a:0?",
                "-c:v",
                "libx264",
                "-preset",
                "fast",
                "-crf",
                "18",
                "-c:a",
                "aac",
                "-t",
                f"{actual_clip_duration:.6f}",
                "-shortest",
                "-movflags",
                "+faststart",
                str(raw_clip_path),
            ],
            check=True,
        )

        temporary_silent_path.unlink(
            missing_ok=True
        )

        clip_tracks = tracks_df[
            (
                tracks_df["frame"]
                >= clip_start_frame
            )
            & (
                tracks_df["frame"]
                <= actual_clip_end_frame
            )
        ].copy()

        create_clip_cvat_xml(
            clip_tracks=clip_tracks,
            clip_start_frame=clip_start_frame,
            clip_total_frames=frames_written,
            clip_name=clip_stem,
            xml_path=xml_path,
        )

        track_ids = sorted(
            clip_tracks["track_id"]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        manifest_rows.append(
            {
                "video_id": video_id,
                "bark_id": bark_id,
                "source_video_path": str(
                    source_video_path
                ),
                "bark_start_time": bark_start,
                "bark_end_time": bark_end,
                "clip_start_time": clip_start_time,
                "clip_end_time": (
                    clip_start_time
                    + actual_clip_duration
                ),
                "clip_start_frame": clip_start_frame,
                "clip_end_frame": (
                    actual_clip_end_frame
                ),
                "clip_frame_count": frames_written,
                "raw_clip_path": str(raw_clip_path),
                "cvat_xml_path": str(xml_path),
                "track_ids_in_clip": "|".join(
                    map(str, track_ids)
                ),
                "barking_track_id": "",
                "label": "",
                "notes": "",
            }
        )

        print(
            f"Created: {raw_clip_path.name} "
            f"+ {xml_path.name}"
        )

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(
        manifest_csv_path,
        index=False,
    )

    print("\nCVAT-ready generation complete")
    print("Output directory:", output_dir.resolve())
    print("Manifest:", manifest_csv_path.resolve())
    print("Number of clip/XML pairs:", len(manifest_df))

    return manifest_df

In [15]:
from pathlib import Path
import pandas as pd

VIDEO_ID = 16
CSV_PATH = Path("validation_segments.csv")

# Set this to True only when you want to rerun
# the expensive full-video RT-DETR tracking.
RERUN_TRACKING = False

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Missing CSV: {CSV_PATH.resolve()}"
    )

segments_df = pd.read_csv(CSV_PATH)

video_segments = (
    segments_df[
        segments_df["video_id"] == VIDEO_ID
    ]
    .sort_values(["start_time", "end_time"])
    .reset_index(drop=True)
)

if video_segments.empty:
    raise ValueError(
        f"No bark segments found for video_id={VIDEO_ID}"
    )

csv_video_path = Path(
    str(video_segments.iloc[0]["video_path"])
)

original_filename = csv_video_path.name

candidate_paths = [
    csv_video_path,
    Path("vids") / f"{VIDEO_ID}_{original_filename}",
    Path("vids") / original_filename,
]

source_video = next(
    (
        path
        for path in candidate_paths
        if path.exists()
    ),
    None,
)

if source_video is None:
    raise FileNotFoundError(
        "Could not find source video. Checked:\n"
        + "\n".join(
            str(path.resolve())
            for path in candidate_paths
        )
    )

output_root = (
    Path("bark_annotation_outputs")
    / str(VIDEO_ID)
)

boxed_preview_video = (
    output_root
    / f"{VIDEO_ID}_full_boxed.mp4"
)

tracks_csv = (
    output_root
    / f"{VIDEO_ID}_tracks.csv"
)

cvat_ready_dir = output_root / "cvat_ready"

manifest_csv = (
    output_root
    / f"{VIDEO_ID}_cvat_manifest.csv"
)

output_root.mkdir(
    parents=True,
    exist_ok=True,
)

cvat_ready_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Source video:", source_video.resolve())
print("Bark segments:", len(video_segments))
print("Tracks CSV:", tracks_csv.resolve())

# You already generated 16_tracks.csv, so this
# should normally skip the expensive tracking step.
if RERUN_TRACKING or not tracks_csv.exists():
    print(
        "\nRunning full-video RT-DETR tracking..."
    )

    tracks_df = run_full_video_annotation_tracking(
        video_id=VIDEO_ID,
        video_path=source_video,
        boxed_video_path=boxed_preview_video,
        tracks_csv_path=tracks_csv,
        model=model,
        appearance_model=appearance_model,
        appearance_threshold=appearance_threshold,
        appearance_ema=appearance_ema,
        conf_threshold=conf_threshold,
        iou_threshold=iou_threshold,
        max_age=max_age,
        device=device,
    )
else:
    print(
        "\nUsing existing full-video tracks CSV."
    )

    tracks_df = pd.read_csv(tracks_csv)

print(
    "\nCreating clean clips and matching "
    "CVAT XML annotations...\n"
)

manifest_df = make_cvat_ready_bark_clips(
    video_id=VIDEO_ID,
    segments_csv_path=CSV_PATH,
    source_video_path=source_video,
    tracks_csv_path=tracks_csv,
    output_dir=cvat_ready_dir,
    manifest_csv_path=manifest_csv,
    pad_before=0.5,
    pad_after=0.5,
    label_name="Dog",
)

print("\nDONE")
print("CVAT-ready folder:", cvat_ready_dir.resolve())
print("Manifest:", manifest_csv.resolve())

for _, row in manifest_df.iterrows():
    print()
    print("Video:", Path(row["raw_clip_path"]).resolve())
    print("XML:  ", Path(row["cvat_xml_path"]).resolve())

display(manifest_df)

Source video: /supernova/data/home/aryah/vids/16_7O82jDnyJdc.mp4
Bark segments: 14
Tracks CSV: /supernova/data/home/aryah/bark_annotation_outputs/16/16_tracks.csv

Running full-video RT-DETR tracking...
Successfully loaded imagenet pretrained weights from "/supernova/data/home/aryah/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Processed 500 frames (16.6s)
Processed 1000 frames (33.3s)
Processed 1500 frames (50.0s)
Processed 2000 frames (66.6s)
Processed 2500 frames (83.3s)
Processed 3000 frames (100.0s)
Saved tracks CSV: bark_annotation_outputs/16/16_tracks.csv (3543 rows)


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Saved boxed video: bark_annotation_outputs/16/16_full_boxed.mp4

Creating clean clips and matching CVAT XML annotations...

Created: 16_bark_000.mp4 + 16_bark_000.xml
Created: 16_bark_001.mp4 + 16_bark_001.xml
Created: 16_bark_002.mp4 + 16_bark_002.xml
Created: 16_bark_003.mp4 + 16_bark_003.xml
Created: 16_bark_004.mp4 + 16_bark_004.xml
Created: 16_bark_005.mp4 + 16_bark_005.xml
Created: 16_bark_006.mp4 + 16_bark_006.xml
Created: 16_bark_007.mp4 + 16_bark_007.xml
Created: 16_bark_008.mp4 + 16_bark_008.xml
Created: 16_bark_009.mp4 + 16_bark_009.xml
Created: 16_bark_010.mp4 + 16_bark_010.xml
Created: 16_bark_011.mp4 + 16_bark_011.xml
Created: 16_bark_012.mp4 + 16_bark_012.xml
Created: 16_bark_013.mp4 + 16_bark_013.xml

CVAT-ready generation complete
Output directory: /supernova/data/home/aryah/bark_annotation_outputs/16/cvat_ready
Manifest: /supernova/data/home/aryah/bark_annotation_outputs/16/16_cvat_manifest.csv
Number of clip/XML pairs: 14

DONE
CVAT-ready folder: /supernova/data/home

,video_id,bark_id,source_video_path,bark_start_time,bark_end_time,clip_start_time,clip_end_time,clip_start_frame,clip_end_frame,clip_frame_count,raw_clip_path,cvat_xml_path,track_ids_in_clip,barking_track_id,label,notes
0,16,0,vids/16_7O82jDnyJdc.mp4,6.488,6.936,5.966667,7.466667,179,223,45,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,0|1|2,,,
1,16,1,vids/16_7O82jDnyJdc.mp4,28.184,28.312,27.666667,28.833333,830,864,35,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,0|1|3,,,
2,16,2,vids/16_7O82jDnyJdc.mp4,29.016,29.336,28.500000,29.866667,855,895,41,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,0|1|3,,,
3,16,3,vids/16_7O82jDnyJdc.mp4,29.976,30.232,29.466667,30.733333,884,921,38,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,1|2|3,,,
4,16,4,vids/16_7O82jDnyJdc.mp4,30.872,31.000,30.366667,31.500000,911,944,34,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,2|3,,,
5,16,5,vids/16_7O82jDnyJdc.mp4,32.216,32.472,31.700000,33.000000,951,989,39,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,2|3,,,
6,16,6,vids/16_7O82jDnyJdc.mp4,44.632,44.952,44.100000,45.466667,1323,1363,41,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,1|3|5,,,
7,16,7,vids/16_7O82jDnyJdc.mp4,46.360,52.824,45.833333,53.333333,1375,1599,225,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,1|4|5,,,
8,16,8,vids/16_7O82jDnyJdc.mp4,53.336,57.240,52.833333,57.766667,1585,1732,148,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,1|4|5,,,
9,16,9,vids/16_7O82jDnyJdc.mp4,57.816,64.152,57.300000,64.666667,1719,1939,221,bark_annotation_outputs/16/cvat_ready/16_bark_...,bark_annotation_outputs/16/cvat_ready/16_bark_...,1|4|5,,,


DONT LOOK PAST THIS

In [ ]:
# Single video processing
warnings.filterwarnings("ignore")

title = "16_7O82jDnyJdc"

video = "vids/16_7O82jDnyJdc.mp4"
segments_csv = "validation_segments.csv"

boxed_output = f"files/{title}-{model_name}-boxed.mp4"
final_output = f"files/examples/{title}-{model_name}-boxed-final.mp4"
cvat_output = f"files/{title}-cvat.xml"

os.makedirs("files/examples", exist_ok=True)

times, log_spec, bark_spec_ids = collect_bark_events(video)

process_video(
    video,
    boxed_output,
    log_spec,
    bark_spec_ids,
    times,
    model,
    appearance_model,
    appearance_threshold,
    appearance_ema,
    conf_threshold,
    iou_threshold,
    max_age,
)

subprocess.run([
    "ffmpeg",
    "-y",
    "-i", boxed_output,
    "-i", video,
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-shortest",
    final_output,
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Boxed video saved to:", final_output)
print("CVAT XML saved to:", f"files/{title}-cvat.xml")

In [ ]:
# --- Verification: boxes drawn/stored at a frame vs boxes visible in the exported JSON ---
# Run AFTER the single-video cell above. Requires `stable_results_per_frame` to be in
# scope: process_video keeps it local, so to verify you can either expose it (e.g.
# `globals()['stable_results_per_frame'] = stable_results_per_frame` before the export
# call) or rely on the JSON-only check below. We reconstruct what Label Studio actually
# shows at FRAME from the JSON sequences and compare against the stored boxes.

FRAME = 1256          # frame to inspect (1256 is already instrumented with a print)
TOL = 1.5             # pixel tolerance when matching stored vs JSON-reconstructed boxes

# Need video dimensions to convert JSON percent coords back to pixels.
_cap = cv2.VideoCapture(video)
W = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
_cap.release()


def _visible_boxes_at(ls_json, frame, w, h):
    """Reconstruct the boxes Label Studio renders at `frame`.

    For each result, take the latest keyframe with kf.frame <= frame. The box is
    visible iff that keyframe is enabled. Interpolation between keyframes is linear,
    but since every stored frame has an explicit enabled keyframe, the at-or-before
    keyframe at a stored frame IS that frame -> exact position.
    """
    out = []
    for pred in ls_json.get("predictions", []):
        for res in pred.get("result", []):
            seq = res["value"]["sequence"]
            prior = [kf for kf in seq if kf["frame"] <= frame]
            if not prior:
                continue
            kf = max(prior, key=lambda k: k["frame"])
            if not kf["enabled"]:
                continue
            x1 = kf["x"] / 100 * w
            y1 = kf["y"] / 100 * h
            x2 = x1 + kf["width"] / 100 * w
            y2 = y1 + kf["height"] / 100 * h
            out.append((res["id"], [x1, y1, x2, y2], kf["frame"]))
    return out


with open(json_output) as f:
    ls_json = json.load(f)

json_boxes = _visible_boxes_at(ls_json, FRAME, W, H)

print(f"Frame {FRAME}: {len(json_boxes)} box(es) visible in Label Studio JSON")
for rid, box, kf_frame in json_boxes:
    exact = " (exact keyframe)" if kf_frame == FRAME else f" (interpolated from frame {kf_frame})"
    print(f"  result {rid}: [{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]{exact}")

# If stable_results_per_frame is available in globals, do a direct stored-vs-JSON match.
store = globals().get("stable_results_per_frame")
if store is not None:
    stored = store.get(FRAME, [])
    print(f"\nFrame {FRAME}: {len(stored)} box(es) stored/drawn in MP4")
    for box, sid in stored:
        print(f"  stable_id {sid}: {box}")

    print("\nMatch check (each stored box should have a JSON box at the same position):")
    json_remaining = [b for _, b, _ in json_boxes]
    for box, sid in stored:
        hit = None
        for jb in json_remaining:
            if all(abs(a - b) <= TOL for a, b in zip(box, jb)):
                hit = jb
                break
        if hit is not None:
            json_remaining.remove(hit)
            print(f"  OK  stable_id {sid} matched JSON box")
        else:
            print(f"  MISS stable_id {sid} box {box} has NO matching JSON box")
    for jb in json_remaining:
        print(f"  PHANTOM JSON box {['%.1f' % v for v in jb]} has NO matching stored box")
    if not json_remaining and all(
        any(all(abs(a - b) <= TOL for a, b in zip(box, jb)) for _, jb, _ in json_boxes)
        for box, _ in stored
    ):
        print("  -> stored and JSON boxes match exactly at this frame")
else:
    print("\n(stable_results_per_frame not in scope — JSON-only check shown above. "
          "Expose it from process_video to enable the stored-vs-JSON comparison.)")


In [ ]:
# Video processing loop
warnings.filterwarnings("ignore")

for video_path in os.listdir(video_folder):
    if video_path.endswith('.mp4'):
        video = os.path.join(video_folder, video_path)
        output = os.path.join(output_folder, video_path.replace('.mp4', '-output.mp4'))

        times, log_spec, bark_spec_ids = collect_bark_events(video)
        process_video(video, 
                      output, 
                      log_spec, 
                      bark_spec_ids, 
                      times, 
                      model, 
                      appearance_model, 
                      appearance_threshold, 
                      appearance_ema, 
                      conf_threshold, 
                      iou_threshold, 
                      max_age)

        subprocess.run([
            "ffmpeg",
            "-y",
            "-i", output,
            "-i", video,
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "libx264",
            "-c:a", "aac",
            "-shortest",
            "-crf", "28",
            os.path.join(output_folder, 'finals', video_path.replace('.mp4', '-final.mp4'))
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


### Other

In [ ]:
from ultralytics import YOLO, RTDETR

In [ ]:
video_folder = 'files/data'
output_folder = 'files/rt-outputs'

# Load RT-DETR model
model = RTDETR('rtdetr-l.pt')
model_name = 'rt'
conf_threshold = 0.6

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [ ]:
# Single video processing
warnings.filterwarnings("ignore")

title = "16_7O82jDnyJdc"


video = "vids/16_7O82jDnyJdc.mp4"
segments_csv = "validation_segments.csv"

output = f'files/{title}-{model_name}.mp4'

times, log_spec, bark_spec_ids = collect_bark_events(video)
process_video_old(video,
                  output,
                  log_spec, 
                  bark_spec_ids, 
                  times,
                  model,
                  conf_threshold)

subprocess.run([
            "ffmpeg",
            "-y",
            "-i", output,
            "-i", video,
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            f'files/eval/{title}-{model_name}-final.mp4'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [1]:
# Load dog image
img = Image.open('files/wynnie.jpg').convert('RGB')

# Run inference
results = model(img, device='cpu')


# Draw bounding boxes for dogs
draw = ImageDraw.Draw(img)
dog_count = 0

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])

        if label == 'dog' and conf > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            draw.rectangle([x1, y1, x2, y2], outline='blue', width=4)
            draw.rectangle([x1, y1 - 45, x1 + 200, y1], fill='blue')
            draw.text((x1 + 4, y1 - 40), f'{label} {conf:.2f}', fill='white', font=font)

# Display result
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

NameError: name 'Image' is not defined